# Data Exploration

In [43]:
import pandas as pd
import os
import shutil

# Load the labels file
df = pd.read_csv("../Labels.csv")

In [44]:
# Convert label to binary: GON+ -> 1, GON- -> 0
df['Label_Binary'] = df['Label'].map({'GON+': 1, 'GON-': 0})

# Remove rows with missing important values
df = df.dropna(subset=['Image Name', 'Label', 'Quality Score'])

# Drop column: 'Unnamed: 4'
df = df.drop(columns=['Unnamed: 4'])

## Check Duplicated Labels

In [52]:
# Group by Patient_ID and count unique Label_Binary values
patient_label_check = df.groupby('Patient_ID')['Label_Binary'].nunique().reset_index()
patient_label_check.columns = ['Patient_ID', 'Unique_Labels']

# Mark consistency: True if only one unique label, False if both 0 and 1
patient_label_check['Consistent'] = patient_label_check['Unique_Labels'] == 1

inconsistent_patients = patient_label_check[patient_label_check['Unique_Labels'] > 1]
print("Patients with both 0 and 1 labels:")
print(inconsistent_patients)

Patients with both 0 and 1 labels:
Empty DataFrame
Columns: [Patient_ID, Unique_Labels, Consistent]
Index: []


## Filter Quality >0.6

In [ ]:
# Filter for quality score >= 6
df_filtered = df[df['Quality Score'] >= 6].copy()

# Display the filtered dataframe
print(df_filtered.head())

# Count the binary labels
label_counts = df_filtered['Label_Binary'].value_counts()
print("Count of Label_Binary:")
print(label_counts)

# Create processed folder if not exists
os.makedirs('../data/processed/filter_0.6', exist_ok=True)

# Filter and copy images
for image in df_filtered['Image Name']:
    src = f'../data/raw/{image}'
    dst = f'../data/processed/filter_0.6/{image}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {image}')
    else:
        print(f'Image {image} not found')

print("Image filtering and copying completed.")

# Save the filtered dataframe
df_filtered.to_csv('../data/processed/Labels_0.6.csv', index=False)
print("Saved df_filtered to ../data/processed/Labels_0.6.csv")

   Image Name  Patient Label  Quality Score  Label_Binary
0     0_0.jpg        0  GON+           6.18             1
8     5_0.jpg        5  GON+           6.58             1
9     5_1.jpg        5  GON+           6.67             1
10    6_0.jpg        6  GON+           7.04             1
11    6_1.jpg        6  GON+           6.19             1
Count of Label_Binary:
Label_Binary
1    306
0    128
Name: count, dtype: int64
Copied 0_0.jpg
Copied 5_0.jpg
Copied 5_1.jpg
Copied 6_0.jpg
Copied 6_1.jpg
Copied 10_0.jpg
Copied 12_0.jpg
Copied 12_2.jpg
Copied 12_3.jpg
Copied 12_4.jpg
Copied 13_0.jpg
Copied 14_0.jpg
Copied 16_1.jpg
Copied 16_2.jpg
Copied 17_0.jpg
Copied 17_1.jpg
Copied 17_2.jpg
Copied 17_3.jpg
Copied 17_4.jpg
Copied 17_5.jpg
Copied 18_0.jpg
Copied 18_1.jpg
Copied 18_2.jpg
Copied 18_4.jpg
Copied 20_1.jpg
Copied 21_0.jpg
Copied 21_1.jpg
Copied 22_0.jpg
Copied 22_1.jpg
Copied 23_0.jpg
Copied 23_1.jpg
Copied 24_0.jpg
Copied 26_0.jpg
Copied 26_1.jpg
Copied 27_1.jpg
Copied 28_1.jpg
C

## Preprocess

In [46]:
import os
import cv2
import numpy as np

def preprocess_glaucoma_image(image_path, output_size=(224, 224), use_clahe=True):
    """
    Preprocess a glaucoma fundus image by:
    1. Loading image
    2. Removing black borders by cropping retina area
    3. Padding to square shape
    4. Resizing to standard size
    5. Optionally applying mild CLAHE contrast enhancement
    """

    # Step 1: Load Image
    original_image = cv2.imread(image_path)
    if original_image is None:
        raise ValueError(f"Image not found: {image_path}")

    # Step 2: Convert to grayscale
    gray_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)

    # Step 3: Threshold to detect retina region
    _, binary_mask = cv2.threshold(gray_image, 10, 255, cv2.THRESH_BINARY)

    # Step 4: Find retina contour
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise ValueError("No retina region detected in the image.")

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)

    # Crop retina
    cropped_image = original_image[y:y+h, x:x+w]

    # Step 5: Pad to square
    height, width = cropped_image.shape[:2]

    if height > width:
        pad_total = height - width
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        padded_image = cv2.copyMakeBorder(
            cropped_image,
            0, 0,
            pad_left, pad_right,
            cv2.BORDER_CONSTANT,
            value=[0, 0, 0]
        )
    elif width > height:
        pad_total = width - height
        pad_top = pad_total // 2
        pad_bottom = pad_total - pad_top
        padded_image = cv2.copyMakeBorder(
            cropped_image,
            pad_top, pad_bottom,
            0, 0,
            cv2.BORDER_CONSTANT,
            value=[0, 0, 0]
        )
    else:
        padded_image = cropped_image

    # Step 6: Resize
    resized_image = cv2.resize(padded_image, output_size, interpolation=cv2.INTER_AREA)

    # Step 7: Optional mild CLAHE
    if use_clahe:
        lab = cv2.cvtColor(resized_image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)

        lab = cv2.merge((l, a, b))
        resized_image = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    return resized_image

# Process all images in a directory
def preprocess_images_in_directory(input_dir, output_dir, output_size=(224, 224), use_clahe=True):
    """
    Process all fundus images in input_dir and save them into output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        image_path = os.path.join(input_dir, filename)

        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            try:
                preprocessed_image = preprocess_glaucoma_image(
                    image_path,
                    output_size=output_size,
                    use_clahe=use_clahe
                )

                output_path = os.path.join(output_dir, filename)
                cv2.imwrite(output_path, preprocessed_image)

                print(f"Preprocessed and saved: {filename}")

            except Exception as e:
                print(f"Error processing {filename}: {e}")

In [47]:
input_directory = r"../data/processed/filter_0.6"
output_directory = r"../data/processed/preprocessed_glaucoma"

preprocess_images_in_directory(
    input_dir=input_directory,
    output_dir=output_directory,
    output_size=(224, 224),   # for ResNet / EfficientNet
    use_clahe=True
)

Preprocessed and saved: 0_0.jpg
Preprocessed and saved: 100_2.jpg
Preprocessed and saved: 100_4.jpg
Preprocessed and saved: 100_5.jpg
Preprocessed and saved: 100_6.jpg
Preprocessed and saved: 100_7.jpg
Preprocessed and saved: 100_8.jpg
Preprocessed and saved: 102_1.jpg
Preprocessed and saved: 103_0.jpg
Preprocessed and saved: 104_1.jpg
Preprocessed and saved: 104_3.jpg
Preprocessed and saved: 105_0.jpg
Preprocessed and saved: 107_0.jpg
Preprocessed and saved: 108_0.jpg
Preprocessed and saved: 108_1.jpg
Preprocessed and saved: 108_3.jpg
Preprocessed and saved: 109_0.jpg
Preprocessed and saved: 109_1.jpg
Preprocessed and saved: 109_3.jpg
Preprocessed and saved: 10_0.jpg
Preprocessed and saved: 110_0.jpg
Preprocessed and saved: 110_1.jpg
Preprocessed and saved: 112_2.jpg
Preprocessed and saved: 113_0.jpg
Preprocessed and saved: 116_0.jpg
Preprocessed and saved: 116_1.jpg
Preprocessed and saved: 117_0.jpg
Preprocessed and saved: 118_0.jpg
Preprocessed and saved: 119_1.jpg
Preprocessed and 